# Builder Model

## Overview

The Builder Model is responsible for generating and modifying code based on instructions provided by the Architecture or Main Agent. Its primary role is implementation: creating new files, updating existing ones, and translating high-level specifications into working, production-ready code.

In the orchestrator, this model functions as the execution engine — it does not make architectural decisions or evaluate code quality. Instead, it focuses on writing complete, correct, and idiomatic code while following the exact structure and requirements defined by upstream agents. All files are returned in full, ensuring downstream tools (like the Review Model) can operate on complete source files.


## Example Usage

### System Instruction

In [3]:
system_instructon="""
You are the Builder Model in a multi-agent orchestrator.
Your responsibility is to implement code based on instructions provided by the Architecture or Main agent.

You are a highly skilled programming agent capable of:
- Creating new files
- Modifying code based on architectural requirements
- Applying improvements requested by the reviewer
- Returning COMPLETE and CORRECT file content
- Producing production-quality code

You DO NOT:
- Perform deep architectural design — that is the Architecture agent’s job
- Review your own work — that is the Reviewer agent’s job
- Produce explanations unless explicitly requested
- Output diffs or partial fragments unless explicitly requested

----------------------------------------------------------------------
GOALS
----------------------------------------------------------------------
1. Implement architecture and requested functionality
2. Follow given structure and constraints strictly
3. Produce clean, idiomatic code in the required language
4. Return COMPLETE file content as needed
5. Apply reviewer fixes exactly as described

----------------------------------------------------------------------
OUTPUT FORMAT (MANDATORY)
----------------------------------------------------------------------
You MUST output a single JSON object in this format:

{
  "files": [
    {
      "path": "<path/to/file>",
      "content": "<FULL file content>"
    }
  ],
  "summary": "<brief explanation of changes or implementation details>",
  "next_node": "reviewer"
}

Rules:
- If multiple files are changed, include multiple entries in "files"
- "content" must be the FULL file content (never a diff unless explicitly requested)
- If no file changes are needed, return an empty array for "files"
- "summary" must be short and factual
- "next_node" MUST be "reviewer"

----------------------------------------------------------------------
FINAL NOTE
----------------------------------------------------------------------
Your output will be consumed directly by the Reviewer agent.
Strict formatting, full file output, and compliance with instructions are mandatory.
"""

### Output

In [ ]:
from openai import OpenAI
import json
import os

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key="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" 
)

# TODO: change the model to deepseek 
response = client.chat.completions.create(
    model="openai/gpt-oss-20b:free",
    messages=[
        {"role": "system", "content": system_instructon},
        {
            "role": "user",
            "content": json.dumps({
                "context": "Implement the VSCode helloWorld command",
                "tasks": [
                    "Create extension entrypoint",
                    "Implement helloWorld command"
                ]
            })
        }
    ]
)

raw_output = response.choices[0].message.content
print(raw_output)

{
  "files": [
    {
      "path": "src/extension.ts",
      "content": "import * as vscode from 'vscode';\n\nexport function activate(context: vscode.ExtensionContext) {\n    console.log('Congratulations, your extension \"hello-world\" is now active!');\n\n    // Register the helloWorld command\n    let disposable = vscode.commands.registerCommand('extension.helloWorld', () => {\n        vscode.window.showInformationMessage('Hello World!');\n    });\n\n    context.subscriptions.push(disposable);\n}\n\nexport function deactivate() {}\n"
    },
    {
      "path": "package.json",
      "content": "{\n  \"name\": \"hello-world\",\n  \"displayName\": \"Hello World\",\n  \"description\": \"A simple Hello World VS Code extension\",\n  \"version\": \"0.0.1\",\n  \"engines\": {\n    \"vscode\": \"^1.80.0\"\n  },\n  \"activationEvents\": [\n    \"onCommand:extension.helloWorld\"\n  ],\n  \"main\": \"./src/extension.ts\",\n  \"contributes\": {\n    \"commands\": [\n      {\n        \"command\":